# YOLO11m 학습 노트북

알약 사진에서 **알약 위치(bbox)와 약 종류(56종)**를 함께 찾는 모델을 YOLO11m으로 학습한다.
위에서 아래로 실행하면 **전처리 → 학습 → 검증 → 오류 분석**까지 진행된다.

## 왜 YOLO11인가 (참고 논문 요약)

참고: *YOLO11 vs AutoML 알약 검출 비교* (PMC13068000)

| 논문에서 확인한 점 | 우리 프로젝트에 주는 의미 |
|---|---|
| YOLO11은 **1단계 검출기**: 사진 전체를 한 번 순전파해서 박스와 클래스 확률을 함께 예측 | 사진 한 장에서 알약 2~4개의 위치와 종류를 동시에 찾는 우리 과제에 그대로 맞음 |
| 코드 기반이라 **모델 구조와 학습 파라미터를 직접 제어**할 수 있음(AutoML은 맞춤화가 제한적) | 분할 규칙, 증강, NMS를 우리 데이터에 맞게 정할 수 있음 |
| **플랫폼 비용 0**, 엣지 배포에 유리 | Colab이나 개인 GPU만으로 실험 가능 |
| 학습 데이터가 늘수록 **성능이 가장 일관되게, 거의 선형으로 오름** | 나중에 외부 데이터를 넣으면 효과를 기대할 수 있음 |
| 가장 까다로운 다중 알약 데이터셋에서 향상 폭이 가장 컸음(재현율 0.49 → 0.91) | 우리처럼 한 사진에 여러 알약이 있는 경우에도 데이터를 늘리면 개선 여지가 큼 |
| 대신 최적 성능에는 **더 큰 학습 데이터**와 학습 시간이 필요함 | 우리 데이터(232장)만으로는 한계가 있음 → 외부 데이터 계획과 연결됨 |

## 왜 m(medium) 크기인가

논문도 **속도·정확도·학습 시간의 균형**을 이유로 YOLO11m을 썼다. 우리 상황에서도 같은 판단이 맞는다.

| 기준 | 우리 상황 | YOLO11m이 맞는 이유 |
|---|---|---|
| **어려운 부분** | 위치 찾기는 쉬움(배경이 균일). **어려운 건 약 종류 구분**: 색·모양이 비슷하고 각인으로만 갈리는 약이 많음 | n·s보다 표현력이 커서 각인이나 색조 같은 미세한 차이를 구분하기 유리함 |
| **데이터 양** | 지금 232장, 나중에 외부 데이터 추가 예정 | COCO 사전학습 가중치에서 시작(전이학습)해 적은 데이터로도 학습 가능. 데이터가 늘면 큰 모델일수록 이득이 큼. 나중에 ai_hub에서 추가 데이터를 받을 수 있기에 이러한 면에서도 적합 |
| **알약 크기** | 원본 976×1280에서 약 150~450px, 640으로 줄여도 약 100~300px | 작은 물체가 아니라 기본 입력 640으로 충분 |
| **학습 환경** | Colab T4(16GB) 이상 | 파라미터 약 2천만 개로 T4에서 batch 16 학습 가능 |

| 모델 | 파라미터 | COCO mAP50-95 | 이 프로젝트에서 |
|---|---:|---:|---|
| YOLO11n | 2.6M | 39.5 | 가장 빠름. 비교 기준선 |
| YOLO11s | 9.4M | 47.0 | 중간 |
| **YOLO11m** | **20.1M** | **51.5** | **선택**: Colab GPU로 감당할 수 있는 범위에서 구분 능력이 크게 오름 |
| YOLO11l | 25.3M | 53.4 | m 대비 이득은 작고 학습은 느림 |

### 주의할 점

- **과적합 위험**: 232장만으로는 큰 모델이 사진을 외울 수 있다. 대응책은 세 가지다.
  ① 세팅 묶음 단위 분할로 val 점수가 부풀려지지 않게 한다.
  ② 증강으로 장면 암기를 줄인다.
  ③ 같은 분할에서 n/s도 학습해 비교한다(`CFG["weights"]`만 바꾸면 된다).
- **학습 시간**: 논문 말대로 m은 학습이 오래 걸린다. 동작 확인은 `epochs=1`로 짧게 하고, 전체 학습은 GPU에서 돌린다.
- **팀 문서**: `AGENTS.md`에는 아직 "검출 모델 미정"으로 되어 있다. 확정되면 문서도 갱신한다.

## 전체 흐름

```
원본 json (data/raw)                         외부 데이터 (나중에, 선택)
      │                                            │
      ▼                                            ▼
[2] 박스 표로 읽기 + EDA 보완 반영            [4] 박스 표로 읽기
      ▼                                            │
[3] train / val 분할 × 2 (세팅 / 조합 기준)         ▼
      │                                  누수 필터 (val과 같은 장면 제거)
      └──────────────┬─────────────────────────────┘
                     ▼   val = 우리 데이터만 / train = 우리 train + 외부
[5] YOLO 형식 변환 (분할 방식마다 1개): images/, labels/(.txt), data.yaml
                     ▼
[6] YOLO11m 학습 (실험 목록에서 분할 방식·하이퍼파라미터를 골라 실행)
                     ▼
[7] 표준 지표: mAP50, mAP50-95, 약별 AP (+ 누수 없는 val 사진만의 점수)
[8] 오류 양상 (논문 방식, conf 0.5): 정밀도·재현율·FNR·OER + 착각 표
[9] (선택) 데이터 양별 성능 곡선: 외부 데이터를 넣을 가치가 있는지 판단
```

**핵심은 "같은 표 형식"이다.** 우리 데이터든 외부 데이터든 먼저 `한 행 = 박스 하나` 표로 바꾸면,
그 뒤 단계(분할, 변환, 학습)는 출처와 관계없이 똑같이 돌아간다. 외부 데이터를 붙일 때 고칠 곳은 1번 설정의 경로뿐이다.

| 규칙 | 이유 |
|---|---|
| 같은 **묶음**은 train과 val 중 한쪽에만 (묶는 기준: 세팅 또는 조합) | 형제 사진이 양쪽에 있으면 점수가 부풀려짐 |
| 박스 5개 이하인 **희귀한 약**의 묶음은 train 고정 | val로 보내면 배울 데이터가 사라짐 |
| **val은 우리 데이터만**, 외부 데이터는 train에만 | 대회 test와 같은 조건에서 평가 |
| **증강은 train에만** | val은 실제 사진 그대로 평가 |

## 0. 환경 준비 (Colab)

**Colab 런타임 준비(저장소 받기, 패키지 설치, 데이터 받기)는 팀 공용 `colab.ipynb`가 맡는다.**
새 런타임이면 `colab.ipynb`를 위에서부터 먼저 실행하고, 이 노트북은 **같은 Colab 서버**에 연결해서 연다.

이 셀은 아무것도 설치하지 않는다. 아래 두 가지만 한다.

1. Colab이면 저장소 폴더(`/content/pill-bbox`)로 들어간다. 로컬에서는 아무것도 하지 않는다.
2. 이 노트북에 필요한 패키지와 데이터가 있는지 **점검만** 하고, 빠진 게 있으면 무엇을 실행하면 되는지 알려 준다.

> 이 셀이 오류로 멈추면 **아래 셀을 실행하지 않는다.** 앞 셀이 실패한 채로 아래를 실행하면 엉뚱한 `NameError`가 연달아 난다.

In [7]:
import importlib.util
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COLAB_ROOT = Path("/content/pill-bbox")
# import 이름: 설치 이름. 이 노트북이 쓰는 패키지
REQUIRED = {
    "pandas": "pandas",
    "sklearn": "scikit-learn",
    "yaml": "pyyaml",
    "ultralytics": "ultralytics",
    "dotenv": "python-dotenv",
    "wandb": "wandb",
}

if IN_COLAB:
    if not COLAB_ROOT.exists():
        raise FileNotFoundError("저장소가 없다 → colab.ipynb의 2번(저장소 받기)을 먼저 실행한다.")
    os.chdir(COLAB_ROOT)  # 이후 셀은 저장소 폴더에서 실행된다

missing = [pkg for mod, pkg in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if missing:
    raise ModuleNotFoundError(
        f"패키지 없음: {missing} → colab.ipynb의 3번(패키지 설치)을 먼저 실행한다."
    )
if IN_COLAB and not (COLAB_ROOT / "data" / "raw" / "train_images").exists():
    raise FileNotFoundError("원본 데이터가 없다 → colab.ipynb의 5번(데이터 받기)을 먼저 실행한다.")

print("작업 위치:", Path.cwd())
print("Python:", sys.version.split()[0], "| 패키지 점검 통과")

작업 위치: /content/pill-bbox
Python: 3.13.16 | 패키지 점검 통과


### 🔍 코드 풀이: 0. 환경 준비

> 비유: 요리 시작 전 **주방 점검표**. 재료를 직접 사 오지는 않고, 빠진 게 있으면 "장보기 담당(`colab.ipynb`)한테 가라"고 알려 준다.

| 코드 | 쉽게 말하면 |
|---|---|
| `IN_COLAB = "google.colab" in sys.modules` | "지금 Colab에서 돌고 있나?"를 확인하는 스위치. Colab이면 `True` |
| `REQUIRED = {...}` | 점검표: 왼쪽은 코드에서 부르는 이름(`import sklearn`), 오른쪽은 설치할 때 이름(`scikit-learn`) |
| `os.chdir(COLAB_ROOT)` | 작업실(`/content/pill-bbox`) 문을 열고 들어간다. 터미널의 `cd`와 같음 |
| `importlib.util.find_spec(mod)` | 패키지가 **설치돼 있는지만** 살짝 확인한다. 실제로 불러오지는 않아 빠르다 |
| `raise ...Error(...)` | 빠진 게 있으면 **여기서 멈추고** 무엇을 하면 되는지 알려 준다. 아래 셀이 엉뚱한 오류를 내는 걸 막는다 |

마지막에 `패키지 점검 통과`가 나오면 다음 셀로 간다.

## 1. 설정

설정을 **바뀌었을 때 다시 실행해야 하는 범위**에 따라 세 묶음으로 나눈다.

| 묶음 | 내용 | 바꾸면 |
|---|---|---|
| `DATA_CFG` | train/val 분할 규칙 (분할 방식, fold, 세팅 묶음 기준, 희귀 약 기준) | 3~5단계(분할·변환)부터 다시 실행 |
| `TRAIN_CFG` | 모델과 학습 하이퍼파라미터, 증강 | 6단계(학습)만 다시 실행 |
| `EVAL_CFG` | 오류 분석 기준 (논문 방식 conf 0.5, IoU 0.5) | 8단계만 다시 실행 |

`TRAIN_CFG`는 **모든 실험의 기본값**이다. 실험마다 다른 값은 여기를 고치지 않고, 6단계에서 **바꿀 값만** 적어 덮어쓴다.
키 이름은 ultralytics `model.train()`의 인자 이름과 같게 맞춰 두었기 때문에, 새 하이퍼파라미터도 키 하나만 추가하면 된다.
외부 데이터를 쓸 때는 `EXTERNAL_ANN_DIR`, `EXTERNAL_IMG_DIR`만 채운다.

In [8]:
import sys
from pathlib import Path

# 노트북이 어디서 실행되든 프로젝트 루트(pyproject.toml이 있는 폴더)를 찾는다 (로컬·Colab 공통)
PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), None
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"프로젝트 루트를 못 찾았다 (지금 위치: {Path.cwd()}). 0번 환경 준비 셀을 먼저 실행한다."
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TRAIN_IMG_DIR = RAW_DIR / "train_images"
ANN_DIR = RAW_DIR / "train_annotations"
ADDED_PATH = PROCESSED_DIR / "added_annotations.csv"  # EDA에서 새로 추가한 박스
FIXED_PATH = PROCESSED_DIR / "fixed_annotations.csv"  # EDA에서 위치를 고친 박스
RUNS_DIR = PROJECT_ROOT / "runs"  # 학습 결과와 가장 좋은 가중치를 모아 두는 곳 (git 제외)

SEED = 42  # 분할과 학습 모두 이 값으로 고정해 같은 결과를 다시 만들 수 있게 한다

# 데이터·분할 설정: 바꾸면 3~5단계부터 다시 실행
DATA_CFG = {
    "split_modes": ["setup", "combo"],  # 만들 데이터셋: 세팅 기준 / 조합 기준
    "pin_rare": True,  # 희귀한 약의 묶음은 train 고정 (두 방식에 똑같이 적용)
    "fold": 0,  # 0 ~ n_splits-1. 이번 실험의 val로 쓸 묶음 번호
    "n_splits": 5,
    "same_dist": 30,  # 세팅 공유 판정: 박스 중심 거리(px)
    "min_same": 2,  # 세팅 공유 판정: 같은 자리여야 하는 공통 약 수
    "rare_max": 5,  # 박스가 이 개수 이하인 약 → 그 묶음은 train 고정
}

# 학습 기본값: 모든 실험의 출발점. 키 이름 = ultralytics train() 인자 이름
TRAIN_CFG = {
    "weights": "yolo11m.pt",  # 시작 가중치. 크기 비교는 yolo11n.pt / yolo11s.pt
    "epochs": 100,
    "imgsz": 640,
    "batch": 16,  # T4 기준. -1이면 GPU 메모리에 맞춰 자동
    "patience": 30,  # val 성능이 이만큼 epoch 동안 안 오르면 조기 종료
    "optimizer": "auto",  # "auto"면 lr0를 무시하고 자동 선택. lr0를 실험하려면 "AdamW" 등으로
    "lr0": 0.01,
    "seed": SEED,
    # 증강 (train에만 적용)
    "hsv_h": 0.005,  # 색조: 색이 약 구분 단서라 아주 약하게
    "hsv_s": 0.5,
    "hsv_v": 0.3,  # 밝기: 가장자리 조명 얼룩 대비
    "degrees": 10.0,  # 회전
    "flipud": 0.5,
    "fliplr": 0.5,
    "mosaic": 1.0,
    "close_mosaic": 10,  # 마지막 10 epoch은 mosaic 끔
}

# 오류 분석 기준 (논문 관례)
EVAL_CFG = {
    "conf": 0.5,  # 이 신뢰도 이상인 예측만 확정
    "iou": 0.5,  # 정답 박스와 이만큼 겹치면 같은 위치로 봄
}

# 외부 데이터 (쓰지 않으면 None). 우리 데이터와 같은 COCO 형식 json이라고 가정한다
EXTERNAL_ANN_DIR = None  # 예: Path("/content/external/annotations")
EXTERNAL_IMG_DIR = None  # 예: Path("/content/external/images")
EXTERNAL_NEW_CLASSES = False  # False면 우리 56종에 없는 약은 외부 데이터에서 뺀다


def yolo_dir(mode: str) -> Path:
    """분할 방식별 YOLO 데이터셋 폴더. 예: data/processed/yolo/setup_fold0"""
    return PROCESSED_DIR / "yolo" / f"{mode}_fold{DATA_CFG['fold']}"


print("프로젝트 루트:", PROJECT_ROOT)
print(f"기본 모델: {TRAIN_CFG['weights']} · fold {DATA_CFG['fold']} · seed {SEED}")
print("분할 방식:", DATA_CFG["split_modes"])
print(
    f"데이터: train 사진 {len(list(TRAIN_IMG_DIR.glob('*.png')))}장 · "
    f"json 폴더 {'있음' if ANN_DIR.exists() else '없음'} · "
    f"EDA 보완 csv {'있음' if ADDED_PATH.exists() and FIXED_PATH.exists() else '없음'}"
)

프로젝트 루트: /content/pill-bbox
기본 모델: yolo11m.pt · fold 0 · seed 42
분할 방식: ['setup', 'combo']
데이터: train 사진 232장 · json 폴더 있음 · EDA 보완 csv 있음


### 🔍 코드 풀이: 1. 설정

> 비유: 요리 전에 **레시피 카드**를 적어 두는 단계. 카드를 세 장으로 나눴다: **장보기 카드**(데이터), **조리법 카드**(학습), **맛 평가 카드**(평가).

**① 프로젝트 루트 찾기**

```python
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if ...), None)
```

지금 폴더에서 **계단을 한 칸씩 올라가며** `pyproject.toml`이라는 **문패**가 붙은 폴더를 찾는다.
끝까지 못 찾으면 `None` → "0번 셀을 먼저 실행하라"는 안내를 띄운다.

**② 경로 만들기**

`PROJECT_ROOT / "data" / "raw"`처럼 `/`로 **주소를 이어 붙인다**. 윈도우·맥·리눅스 어디서든 알맞게 바뀐다.

**③ 카드 세 장**

| 카드 | 비유 | 바꾸면 |
|---|---|---|
| `DATA_CFG` | **장보기 카드**: 어떤 재료를 연습용(train)과 시험용(val)으로 나눌지 | 재료를 다시 나눠야 하니 3단계부터 |
| `TRAIN_CFG` | **조리법 카드**: 불 세기, 조리 시간, 양념 | 요리(학습)만 다시 |
| `EVAL_CFG` | **맛 평가 카드**: 몇 점 이상을 "맛있다"로 칠지 | 평가만 다시 |

**④ 장보기 카드(`DATA_CFG`)의 분할 방식**

| 키 | 뜻 | 비유 |
|---|---|---|
| `split_modes` | 만들 데이터셋 목록. `"setup"`(세팅 기준), `"combo"`(조합 기준) | 반 편성 규칙을 **두 가지**로 해 보고 비교 |
| `pin_rare` | 희귀한 약의 묶음을 train에 고정할지 | 교과서가 한 권뿐인 과목은 **무조건 공부용**으로 |

`yolo_dir("setup")`처럼 방식 이름을 넣으면 그 방식의 데이터셋 폴더 주소를 돌려준다. 두 데이터셋이 **서로 다른 폴더**에 만들어져 섞이지 않는다.

**⑤ 조리법 카드(`TRAIN_CFG`)에서 자주 바꾸는 것**

| 키 | 뜻 | 비유 |
|---|---|---|
| `weights` | 어떤 모델로 시작할지 | 어떤 **선수**를 데려올지 (n·s·m) |
| `epochs` | 전체 사진을 몇 바퀴 공부할지 | 문제집을 **몇 회독** 할지 |
| `batch` | 한 번에 몇 장씩 볼지 | 한 번에 **몇 문제씩** 풀고 채점할지 |
| `patience` | 몇 바퀴 동안 안 늘면 멈출지 | 성적이 **30회 연속** 안 오르면 그만 |
| `lr0` | 한 번에 얼마나 크게 고칠지 | 오답 노트를 보고 **습관을 얼마나 확 바꿀지** |
| `hsv_h`, `degrees`, `mosaic` … | 증강 | 같은 문제를 **조금씩 바꿔 낸 변형 문제** |

> ⚠️ `optimizer`가 `"auto"`면 ultralytics가 학습률을 알아서 정하기 때문에 `lr0`를 바꿔도 **효과가 없다**. `lr0`를 실험할 때는 `optimizer`도 `"AdamW"`나 `"SGD"`로 정한다.

**⑥ 이렇게 나눈 이유**

이 카드는 **기본값**이다. 실험할 때는 카드를 고쳐 쓰지 않고, 6단계에서 **"이번엔 이것만 다르게"**라고 메모만 붙인다.

```python
{"name": "lr_low", "optimizer": "AdamW", "lr0": 0.001}  # 나머지는 기본값 그대로
```

키 이름이 ultralytics와 같아서, 새 하이퍼파라미터도 **키 하나만 추가**하면 된다.

**⑦ 마지막 `print`**

요리 전 **재료 점검**: 사진 232장, json 폴더, EDA 보완 csv가 다 있는지 확인한다.

## 2. 박스 표로 읽기

모든 박스를 **한 행 = 박스 하나**인 표(DataFrame)로 바꾼다. 이후 단계(분할, 변환, 학습)는 이 표만 보고 동작한다.

```
json 파일 (사진 1장 + 박스 여러 개)          박스 표
┌───────────────────────────┐          file_name | category_id | x | y | w | h | ...
│ images: [{file_name, ...}]│   →      사진A.png |   3351      | … | … | … | …
│ annotations: [{bbox}, ...]│          사진A.png |  16262      | … | … | … | …
└───────────────────────────┘          사진B.png |   3351      | … | … | … | …
```

- **왜 표로 바꾸나**: 외부 데이터도 같은 표 모양으로 바꾸면, 뒤 단계를 출처와 관계없이 그대로 쓸 수 있다.
- `img_path`: 실제 이미지 위치 / `source`: `ours`(우리 데이터) 또는 `external`(외부 데이터)
- `x, y, w, h`: 박스 왼쪽 위 좌표와 너비·높이 (픽셀, COCO 형식)
- **EDA 보완 반영**: 원본 json은 건드리지 않고, 읽을 때만 위치 수정(`fixed`)과 추가 박스(`added`)를 합친다.
- 정상이라면 **사진 232장 · 약 56종**, EDA 보완은 둘 다 **반영**으로 나온다.

In [9]:
import json

import pandas as pd

BOX_COLS = ["file_name", "img_path", "source", "category_id", "x", "y", "w", "h", "width", "height"]


def load_coco_boxes(ann_dir: Path, img_dir: Path, source: str) -> pd.DataFrame:
    """COCO 형식 json 폴더를 박스 표로 읽는다. 이미지 파일이 없는 박스는 버린다."""
    rows = []
    for path in sorted(ann_dir.rglob("*.json")):
        with open(path, encoding="utf-8") as f:
            data = json.load(f)
        images = {img["id"]: img for img in data["images"]}
        for ann in data["annotations"]:
            img = images[ann["image_id"]]
            x, y, w, h = ann["bbox"]
            rows.append(
                {
                    "file_name": img["file_name"],
                    "img_path": str(img_dir / img["file_name"]),
                    "source": source,
                    "category_id": ann["category_id"],
                    "x": x,
                    "y": y,
                    "w": w,
                    "h": h,
                    "width": img["width"],
                    "height": img["height"],
                }
            )
    boxes = pd.DataFrame(rows, columns=BOX_COLS)
    exists = {p: Path(p).exists() for p in boxes["img_path"].unique()}
    return boxes[boxes["img_path"].map(exists)].reset_index(drop=True)


def load_our_boxes() -> pd.DataFrame:
    """우리 train 박스 + EDA 보완 (위치 수정 → 추가 박스 순서로 반영)."""
    boxes = load_coco_boxes(ANN_DIR, TRAIN_IMG_DIR, source="ours")
    meta = boxes.drop_duplicates("file_name").set_index("file_name")[
        ["img_path", "source", "width", "height"]
    ]
    if FIXED_PATH.exists():
        fixes = pd.read_csv(FIXED_PATH).set_index(["file_name", "category_id"])[
            ["x", "y", "w", "h"]
        ]
        key = pd.MultiIndex.from_frame(boxes[["file_name", "category_id"]])
        target = key.isin(fixes.index)
        boxes.loc[target, ["x", "y", "w", "h"]] = fixes.loc[key[target]].values
    if ADDED_PATH.exists():
        added = pd.read_csv(ADDED_PATH)[["file_name", "category_id", "x", "y", "w", "h"]]
        boxes = pd.concat([boxes, added.join(meta, on="file_name")], ignore_index=True)
    return boxes[BOX_COLS].reset_index(drop=True)


ours = load_our_boxes()
print(
    f"우리 데이터: 사진 {ours['file_name'].nunique()}장 · 박스 {len(ours)}개 · "
    f"약 {ours['category_id'].nunique()}종"
)
print(
    f"EDA 보완: 위치 수정 {'반영' if FIXED_PATH.exists() else '파일 없음'}, "
    f"추가 박스 {'반영' if ADDED_PATH.exists() else '파일 없음'}"
)
ours.head()

우리 데이터: 사진 232장 · 박스 771개 · 약 56종
EDA 보완: 위치 수정 반영, 추가 박스 반영


,file_name,img_path,source,category_id,x,y,w,h,width,height
0,K-001900-016548-019607-029451_0_2_0_2_70_000_2...,/content/pill-bbox/data/raw/train_images/K-001...,ours,1900,644,845,189,190,976,1280
1,K-001900-016548-019607-029451_0_2_0_2_75_000_2...,/content/pill-bbox/data/raw/train_images/K-001...,ours,1900,148,241,181,181,976,1280
2,K-001900-016548-019607-029451_0_2_0_2_90_000_2...,/content/pill-bbox/data/raw/train_images/K-001...,ours,1900,167,248,184,182,976,1280
3,K-001900-016548-019607-029451_0_2_0_2_70_000_2...,/content/pill-bbox/data/raw/train_images/K-001...,ours,16548,144,799,239,239,976,1280
4,K-001900-016548-019607-029451_0_2_0_2_75_000_2...,/content/pill-bbox/data/raw/train_images/K-001...,ours,16548,578,222,233,226,976,1280


### 🔍 코드 풀이: 2. 박스 표로 읽기

> 비유: 흩어진 **영수증(json)** 수백 장을 **엑셀 한 장(표)**으로 옮겨 적는 단계

**① `load_coco_boxes`: json → 표**

1. json 파일을 **한 장씩 연다** (`for path in ...rglob("*.json")`)
2. `images = {img["id"]: img ...}`: 사진 정보를 **번호표 사물함**에 넣어 둔다. 박스의 `image_id`만 보면 어느 사진인지 바로 꺼낼 수 있다.
3. 박스(`annotations`)마다 **한 줄씩** `rows`에 적는다 (사진 이름, 약 코드, 좌표 x·y·w·h)
4. `pd.DataFrame(rows)`: 적은 줄들을 **표**로 만든다
5. 마지막 두 줄: 사진 파일이 실제로 없는 줄은 **지운다** (영수증만 있고 물건이 없는 경우)

**② `load_our_boxes`: 원본 + EDA 보완**

| 단계 | 하는 일 | 비유 |
|---|---|---|
| `load_coco_boxes(...)` | 원본 박스를 읽음 | 원본 영수증 |
| `if FIXED_PATH.exists():` | 같은 사진·같은 약의 박스를 찾아 **좌표를 바꿔 끼움** | 틀린 칸에 **수정테이프** |
| `if ADDED_PATH.exists():` | 빠졌던 박스를 표 **아래에 붙임**. 사진 크기 같은 정보는 `meta`에서 채움 | 빠진 영수증 **추가 첨부** |

원본 json 파일 자체는 **건드리지 않는다**. 읽을 때만 고쳐서 쓴다.

**③ 마지막 출력**

- `ours['file_name'].nunique()`: 사진 이름의 **중복 없는 개수** = 사진 수
- `len(ours)`: 표의 줄 수 = 박스 수
- `ours.head()`: 표의 **첫 5줄 미리보기**

## 3. train / val 분할: 세팅 기준 vs 조합 기준

사진을 **train(공부용)**과 **val(시험용)**으로 나눈다. 핵심은 **비슷한 사진이 양쪽에 갈라지지 않게** 하는 것이다.
"비슷한 사진"을 어디까지로 볼지에 따라 **두 가지 방식**으로 나누고, 둘 다 데이터셋으로 만들어 학습 결과를 비교한다.

### 왜 그냥 무작위로 나누면 안 되나

우리 사진은 같은 알약 배치를 **각도만 바꿔서**(70°, 75°, 90°) 여러 장 찍었다. 또 **알약 일부만 바꿔** 다시 찍은 사진도 있다.
이런 **형제 사진**이 train과 val에 갈라지면, 모델은 시험 문제를 미리 본 셈이 되어 val 점수가 실제보다 높게 나온다.

### 두 가지 방식 (01_eda 8.4절의 후보)

| 방식 | 같은 묶음으로 보는 것 | 01_eda | 막는 누수 | 남는 누수 |
|---|---|---|---|---|
| `combo` (조합 기준) | 같은 조합 = 각도만 다른 사진 | B | 같은 조합 | **세팅 공유**: 알약 일부만 바꿔 찍은 사진 (EDA에서 val의 약 59%) |
| `setup` (세팅 기준) | 같은 조합 + 공통 약 2개 이상이 30px 이내 같은 자리인 다른 조합 | C | 같은 조합 + 세팅 공유 | 없음 |

두 방식 모두 아래 규칙은 똑같이 적용한다. 그래야 **"묶는 단위"만 다른 공정한 비교**가 된다.

- **희귀한 약 지키기**: 박스 5개 이하인 약이 든 묶음은 무조건 train (`DATA_CFG["pin_rare"]`). EDA의 C+에 해당한다.
- **나머지 묶음을 5개 조(fold)로**: `StratifiedGroupKFold`가 묶음을 통째로 옮기면서 각 조에 약 종류가 고르게 들어가게 한다.
  사진에 약이 여러 개라서, 그 사진에서 **가장 희귀한 약**을 대표로 삼아 비율을 맞춘다.

### 비교할 때 주의할 점

두 방식은 **val 사진 자체가 다르다.** 그래서 val 점수를 그대로 비교하면 "모델이 더 잘했다"와 "시험이 더 쉬웠다(누수)"가 섞인다.

- 이 단계에서는 학습 전에 **두 분할의 누수율**을 표로 먼저 본다.
- `combo` 쪽 val 점수가 더 높게 나오면, 그 차이가 **세팅 공유 누수 때문에 부풀려진 몫**일 수 있다.
- 7단계(검증)에서는 각 val에서 **누수 없는 사진만** 골라 점수를 따로 내서, 부풀려진 몫을 확인한다.

In [10]:
import itertools
import warnings

import numpy as np
from sklearn.model_selection import StratifiedGroupKFold

# 대표 약이 사진 1장뿐인 약이 있어 "5개 조에 고르게 못 나눈다"는 경고가 뜬다. 분할 자체는 정상이다
warnings.filterwarnings("ignore", message="The least populated class")


def combo_and_angle(file_name: str) -> tuple[str, int]:
    """파일명 K-003351-016262-041768_0_2_0_2_75_000_200.png → (약 조합, 카메라 각도)."""
    parts = file_name.split("_")
    return parts[0], int(parts[5])


def box_centers(boxes: pd.DataFrame) -> dict:
    """사진마다 {약 코드: 박스 중심 (cx, cy)}."""
    return {
        fn: dict(
            zip(
                g["category_id"],
                zip(g["x"] + g["w"] / 2, g["y"] + g["h"] / 2, strict=True),
                strict=True,
            )
        )
        for fn, g in boxes.groupby("file_name")
    }


def shares_setup(c1: dict, c2: dict) -> bool:
    """두 사진이 세팅을 공유하는가: 공통 약 min_same개 이상의 박스 중심이 same_dist px 이내."""
    n_same = sum(
        np.hypot(c1[c][0] - c2[c][0], c1[c][1] - c2[c][1]) <= DATA_CFG["same_dist"]
        for c in c1.keys() & c2.keys()
    )
    return n_same >= DATA_CFG["min_same"]


def combo_groups(boxes: pd.DataFrame) -> pd.Series:
    """[조합 기준] 사진마다 약 조합 번호를 붙인다. 각도만 다른 사진끼리 같은 묶음."""
    combos = pd.Series({fn: combo_and_angle(fn)[0] for fn in boxes["file_name"].unique()})
    ids = {c: i for i, c in enumerate(sorted(combos.unique()))}
    return combos.map(ids).rename("group")


def setup_groups(boxes: pd.DataFrame) -> pd.Series:
    """[세팅 기준] 사진마다 촬영 세팅 묶음 번호를 붙인다. 같은 조합은 항상 같은 묶음."""
    centers = box_centers(boxes)
    info = pd.DataFrame(
        [(fn, *combo_and_angle(fn)) for fn in centers], columns=["file_name", "combo", "angle"]
    )
    parent = {c: c for c in info["combo"].unique()}  # 처음엔 조합마다 자기 자신이 대표

    def find(c: str) -> str:
        """조합 c가 속한 묶음의 대표 조합을 찾는다."""
        while parent[c] != c:
            parent[c] = parent[parent[c]]
            c = parent[c]
        return c

    for _, g in info.groupby("angle"):  # 같은 각도끼리만 비교
        for r1, r2 in itertools.combinations(g.itertuples(), 2):
            if r1.combo != r2.combo and shares_setup(centers[r1.file_name], centers[r2.file_name]):
                parent[find(r1.combo)] = find(r2.combo)  # 두 묶음을 하나로 합친다
    ids = {root: i for i, root in enumerate(sorted({find(c) for c in parent}))}
    return info.set_index("file_name")["combo"].map(lambda c: ids[find(c)]).rename("group")


def split_train_val(boxes: pd.DataFrame, groups: pd.Series) -> pd.Series:
    """묶음 단위로 사진마다 "train" 또는 "val". pin_rare면 희귀한 약의 묶음은 train 고정."""
    counts = boxes["category_id"].value_counts()
    rarest = (  # 사진 안에서 가장 희귀한 약 = 그 사진의 대표 약
        boxes.assign(n=boxes["category_id"].map(counts))
        .sort_values("n")
        .drop_duplicates("file_name")
        .set_index("file_name")["category_id"]
    )
    pinned = set()  # train에 고정할 묶음
    if DATA_CFG["pin_rare"]:
        rare_ids = counts[counts <= DATA_CFG["rare_max"]].index
        rare_files = boxes.loc[boxes["category_id"].isin(rare_ids), "file_name"].unique()
        pinned = set(groups.loc[rare_files])

    split = pd.Series("train", index=groups.index, name="split")
    free = groups[~groups.isin(pinned)]
    sgkf = StratifiedGroupKFold(n_splits=DATA_CFG["n_splits"], shuffle=True, random_state=SEED)
    folds = list(sgkf.split(free.index, rarest.loc[free.index], groups=free.values))
    _, val_idx = folds[DATA_CFG["fold"]]
    split.loc[free.index[val_idx]] = "val"
    return split


def leak_rate(split: pd.Series, groups: pd.Series) -> float:
    """val 사진 중, 같은 묶음의 사진이 train에 하나라도 있는 사진의 비율."""
    train_groups = set(groups[split == "train"])
    return groups[split == "val"].isin(train_groups).mean()


# 묶음은 두 기준 모두 만든다 (누수율을 서로 교차로 재기 위해)
GROUPS = {"combo": combo_groups(ours), "setup": setup_groups(ours)}
SPLITS = {mode: split_train_val(ours, GROUPS[mode]) for mode in DATA_CFG["split_modes"]}
VAL_FILES = {mode: split[split == "val"].index.tolist() for mode, split in SPLITS.items()}

rows = []
for mode, split in SPLITS.items():
    files_of = {name: split[split == name].index for name in ["train", "val"]}
    cls_of = {
        name: set(ours.loc[ours["file_name"].isin(files), "category_id"])
        for name, files in files_of.items()
    }
    rows.append(
        {
            "분할 방식": mode,
            "묶음 수": GROUPS[mode].nunique(),
            "train 사진": len(files_of["train"]),
            "val 사진": len(files_of["val"]),
            "val 약 종류": len(cls_of["val"]),
            "val에만 있는 약": len(cls_of["val"] - cls_of["train"]),
            "같은 조합 누수": f"{leak_rate(split, GROUPS['combo']):.0%}",
            "같은 세팅 누수": f"{leak_rate(split, GROUPS['setup']):.0%}",
        }
    )
print(f"fold {DATA_CFG['fold']} · 희귀 약 train 고정: {DATA_CFG['pin_rare']}")
pd.DataFrame(rows).set_index("분할 방식")

fold 0 · 희귀 약 train 고정: True


,묶음 수,train 사진,val 사진,val 약 종류,val에만 있는 약,같은 조합 누수,같은 세팅 누수
분할 방식,,,,,,,
setup,66,192,40,17,0,0%,0%
combo,114,194,38,25,0,0%,58%


### 🔍 코드 풀이: 3. 분할

> 비유: 학생(사진)을 **두 반(train·val)**으로 나누는데, **누구를 "같은 반에 둘 사이"로 볼지** 기준을 두 가지로 해 본다.
> - `combo`: **쌍둥이(같은 조합)**만 같은 반
> - `setup`: 쌍둥이 + **짝꿍이었던 친구(세팅 공유)**까지 같은 반

**① 작은 도우미 함수들**

| 함수 | 하는 일 | 비유 |
|---|---|---|
| `combo_and_angle` | 파일명을 `_`로 잘라 약 조합과 각도를 꺼냄 | 학생증에서 **가족 이름과 학년** 읽기 |
| `box_centers` | 사진마다 약별 박스 **가운데 점** 좌표 | 교실에서 **각자 앉은 자리** |
| `shares_setup` | 두 사진에서 같은 약이 **30px 이내 같은 자리**인 게 2개 이상인지 | 같은 자리에 앉은 사람이 2명 이상이면 **같은 반이었던 사이** |

**② 묶음 만들기: 두 가지**

- `combo_groups`: 조합 이름이 같으면 같은 번호. **가족 이름표만 보고** 나눈다.
- `setup_groups`: 조합으로 시작해서, 세팅을 공유하는 조합끼리 **합친다**.
  - `parent`: 각 조합의 **반장** 명부. 처음엔 모두 자기 자신이 반장이다.
  - `find(c)`: 반장의 반장의… 를 따라가 **최종 반장**을 찾는다.
  - 같은 세팅이면 두 무리를 합친다. "친구의 친구는 같은 무리"가 자동으로 된다.

**③ `split_train_val`: 반 나누기** (두 방식이 **같은 함수**를 쓴다 → 묶는 기준만 다름)

1. `rarest`: 사진마다 **가장 희귀한 약**을 대표로 뽑는다
2. `pinned`: 희귀한 약이 든 묶음 → **무조건 train** 명단 (`pin_rare`가 `True`일 때)
3. `StratifiedGroupKFold`: 나머지 묶음을 **통째로** 5개 조로 나누고, `fold`번 조를 val로 표시한다

**④ `leak_rate`: 누수율 재기**

val 사진마다 "**같은 묶음 친구가 train 반에 있나?**"를 보고, 있는 사진의 비율을 낸다.
기준을 바꿔 가며 잰다: 조합 기준으로 재면 `같은 조합 누수`, 세팅 기준으로 재면 `같은 세팅 누수`.

**⑤ 결과 표 읽는 법**

| 칸 | 뜻 | 좋은 값 |
|---|---|---|
| `val 약 종류` | 시험에 나오는 약 수 | 많을수록 넓게 평가 |
| `val에만 있는 약` | train에 0개라 **배울 수 없는** 약 | 0 |
| `같은 조합 누수` | 각도만 다른 쌍둥이가 train에 있는 val 사진 비율 | 두 방식 모두 0% |
| `같은 세팅 누수` | 세팅을 공유한 친구가 train에 있는 val 사진 비율 | `setup`은 0%, `combo`는 0%보다 큼 |

`GROUPS`, `SPLITS`, `VAL_FILES`는 `{"setup": ..., "combo": ...}` 모양의 **사물함**이다. 다음 단계에서 방식 이름으로 꺼내 쓴다.

## 4. 외부 데이터 자리 (train에만 추가)

나중에 ai_hub 같은 외부 데이터를 넣을 자리를 미리 만들어 둔다. 지금은 1단계 설정이 `None`이라 아무것도 더하지 않는다.

**규칙**
- 외부 데이터는 **train에만** 넣는다. val은 항상 우리 데이터만 쓴다. 시험 문제는 바꾸지 않고 공부 자료만 늘린다.
- 아래 세 가지 사진은 뺀다.
  1. 우리 56종에 없는 약이 든 사진. 그 약만 지우면 "사진에 약이 있는데 정답에는 없는" 상태가 되어 모델이 헷갈린다. 그래서 사진을 통째로 뺀다.
  2. 우리 데이터와 파일명이 같은 사진: 중복이다.
  3. val과 겹치는 사진: 커닝이 된다.
     - 조합 기준: val과 **같은 조합**이면 뺀다.
     - 세팅 기준: 같은 조합이거나, **같은 각도에서 약이 같은 자리에 놓인 사진(같은 세팅)**이면 뺀다. 3단계와 같은 판정을 쓴다.
- 분할 방식마다 val이 다르므로, 빼는 사진도 방식마다 따로 계산한다.


In [11]:
def drop_unknown_classes(boxes: pd.DataFrame) -> pd.DataFrame:
    """우리 56종에 없는 약이 하나라도 든 사진은 통째로 뺀다."""
    unknown = ~boxes["category_id"].isin(ours["category_id"].unique())
    bad_files = boxes.loc[unknown, "file_name"].unique()
    return boxes[~boxes["file_name"].isin(bad_files)]


def safe_combo_and_angle(file_name: str) -> tuple[str, int] | None:
    """파일명이 우리 규칙을 따르면 (조합, 각도), 아니면 None."""
    try:
        return combo_and_angle(file_name)
    except (IndexError, ValueError):
        return None


def leaked_files(ext: pd.DataFrame, val_files: list[str], check_setup: bool) -> set[str]:
    """외부 사진 중 val과 겹치는 사진: 같은 조합, (check_setup이면) 같은 세팅."""
    val_combos = {combo_and_angle(fn)[0] for fn in val_files}
    val_centers = box_centers(ours[ours["file_name"].isin(val_files)])
    val_by_angle = {}
    for fn in val_files:
        val_by_angle.setdefault(combo_and_angle(fn)[1], []).append(fn)

    leaked = set()
    for fn, centers in box_centers(ext).items():
        parsed = safe_combo_and_angle(fn)
        if parsed is not None and parsed[0] in val_combos:
            leaked.add(fn)
        elif check_setup:
            # 각도를 알면 같은 각도의 val 사진만, 모르면 val 사진 전부와 비교
            candidates = val_by_angle.get(parsed[1], []) if parsed else val_files
            if any(shares_setup(centers, val_centers[v]) for v in candidates):
                leaked.add(fn)
    return leaked


if EXTERNAL_ANN_DIR is None or EXTERNAL_IMG_DIR is None:
    external = pd.DataFrame(columns=BOX_COLS)
    print("외부 데이터: 설정 없음 → 우리 데이터만으로 학습한다")
else:
    external = load_coco_boxes(Path(EXTERNAL_ANN_DIR), Path(EXTERNAL_IMG_DIR), "external")
    n_raw = external["file_name"].nunique()
    if not EXTERNAL_NEW_CLASSES:
        external = drop_unknown_classes(external)
    n_known = external["file_name"].nunique()
    external = external[~external["file_name"].isin(ours["file_name"])]  # 우리 사진과 같은 파일
    print(
        f"외부 데이터: 사진 {n_raw}장 → 모르는 약 사진 제외 {n_known}장 "
        f"→ 우리와 같은 파일 제외 {external['file_name'].nunique()}장"
    )

# 분할 방식마다 그 방식의 val과 겹치는 외부 사진을 빼고, 남은 것만 train에 더한다
TRAIN_EXTRA = {}
rows = []
for mode in DATA_CFG["split_modes"]:
    leaked = leaked_files(external, VAL_FILES[mode], check_setup=mode == "setup")
    TRAIN_EXTRA[mode] = external[~external["file_name"].isin(leaked)].reset_index(drop=True)
    n_ours = (SPLITS[mode] == "train").sum()
    n_extra = TRAIN_EXTRA[mode]["file_name"].nunique()
    rows.append(
        {
            "분할 방식": mode,
            "val과 겹쳐 뺀 외부 사진": len(leaked),
            "train 사진 (우리)": n_ours,
            "train 사진 (외부)": n_extra,
            "train 사진 합계": n_ours + n_extra,
            "val 사진 (우리만)": len(VAL_FILES[mode]),
        }
    )
pd.DataFrame(rows).set_index("분할 방식")


외부 데이터: 설정 없음 → 우리 데이터만으로 학습한다


,val과 겹쳐 뺀 외부 사진,train 사진 (우리),train 사진 (외부),train 사진 합계,val 사진 (우리만)
분할 방식,,,,,
setup,0,192,0,192,40
combo,0,194,0,194,38


## 5. YOLO 형식으로 바꾸기 (분할 방식마다 하나씩)

YOLO는 정해진 모양의 폴더와 라벨 파일만 읽는다. 3·4단계에서 정한 train/val대로 분할 방식마다 데이터셋을 하나씩 만든다.

    data/processed/yolo/setup_fold0/      ← 세팅 기준
    ├── images/train, images/val           사진 (원본과 하드링크로 연결해서 용량을 더 쓰지 않음)
    ├── labels/train, labels/val           사진마다 txt 한 개: "클래스 cx cy w h"
    ├── data.yaml                          YOLO가 읽는 안내서
    ├── classes.csv                        YOLO 번호 ↔ 약 코드 ↔ 약 이름
    └── split.csv                          어떤 사진이 train/val인지 기록
    data/processed/yolo/combo_fold0/      ← 조합 기준 (구조 같음)

- **클래스 번호표는 두 방식이 똑같다.** 번호가 다르면 두 결과를 비교할 수 없다.
- 라벨 좌표는 사진 크기로 나눈 0~1 값이다. 사진 밖으로 나간 부분은 잘라 낸다.
- 실행할 때마다 폴더를 **지우고 새로 만든다.** fold나 설정을 바꿨을 때 예전 파일이 섞이지 않게 하기 위해서다.


In [12]:
import os
import shutil

import yaml


def class_names(ann_dirs: list[Path]) -> dict[int, str]:
    """json의 categories에서 {약 코드: 약 이름}을 모은다."""
    names = {}
    for ann_dir in ann_dirs:
        for path in ann_dir.rglob("*.json"):
            with open(path, encoding="utf-8") as f:
                names.update({c["id"]: c["name"] for c in json.load(f)["categories"]})
    return names


def link_or_copy(src: Path, dst: Path) -> None:
    """원본 이미지를 하드링크로 연결한다 (용량을 더 쓰지 않음). 안 되면 복사한다."""
    try:
        os.link(src, dst)
    except OSError:
        shutil.copy2(src, dst)


def yolo_lines(img_boxes: pd.DataFrame) -> list[str]:
    """한 사진의 박스를 YOLO 줄로 바꾼다: "클래스 cx cy w h" (사진 크기로 나눈 0~1 값)."""
    lines = []
    for b in img_boxes.itertuples():
        x1, y1 = max(b.x, 0), max(b.y, 0)  # 사진 밖으로 나간 부분은 잘라 낸다
        x2, y2 = min(b.x + b.w, b.width), min(b.y + b.h, b.height)
        if x2 <= x1 or y2 <= y1:
            continue
        cx, cy = (x1 + x2) / 2 / b.width, (y1 + y2) / 2 / b.height
        w, h = (x2 - x1) / b.width, (y2 - y1) / b.height
        lines.append(f"{CLASS_INDEX[b.category_id]} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")
    return lines


def build_yolo_dataset(mode: str) -> Path:
    """분할 방식 하나의 YOLO 데이터셋(images, labels, data.yaml)을 처음부터 새로 만든다."""
    out = yolo_dir(mode)
    if out.exists():
        shutil.rmtree(out)  # 지난 설정으로 만든 파일이 섞이지 않게 매번 새로 만든다
    for name in ["train", "val"]:
        (out / "images" / name).mkdir(parents=True)
        (out / "labels" / name).mkdir(parents=True)

    parts = [ours.assign(split=ours["file_name"].map(SPLITS[mode]))]
    if len(TRAIN_EXTRA[mode]):
        parts.append(TRAIN_EXTRA[mode].assign(split="train"))  # 외부 데이터는 train에만
    boxes = pd.concat(parts, ignore_index=True)

    for (fn, split), g in boxes.groupby(["file_name", "split"]):
        link_or_copy(Path(g["img_path"].iloc[0]), out / "images" / split / fn)
        label = out / "labels" / split / f"{Path(fn).stem}.txt"
        label.write_text("\n".join(yolo_lines(g)) + "\n", encoding="utf-8")

    boxes.drop_duplicates("file_name")[["file_name", "split", "source"]].to_csv(
        out / "split.csv", index=False
    )
    CLASSES.to_csv(out / "classes.csv", index=False)
    cfg = {
        "path": str(out),
        "train": "images/train",
        "val": "images/val",
        "names": {i: str(c) for i, c in enumerate(CLASS_IDS)},
    }
    with open(out / "data.yaml", "w", encoding="utf-8") as f:
        yaml.safe_dump(cfg, f, allow_unicode=True, sort_keys=False)
    return out


# 클래스 번호표: 두 분할 방식이 똑같은 표를 써야 결과를 비교할 수 있다
class_ids = set(ours["category_id"])
if EXTERNAL_NEW_CLASSES:
    class_ids |= set(external["category_id"])
CLASS_IDS = sorted(int(c) for c in class_ids)
CLASS_INDEX = {c: i for i, c in enumerate(CLASS_IDS)}  # 약 코드 → YOLO 클래스 번호 (0부터)
ann_dirs = [ANN_DIR] + ([Path(EXTERNAL_ANN_DIR)] if EXTERNAL_ANN_DIR else [])
names = class_names(ann_dirs)
CLASSES = pd.DataFrame(
    {
        "yolo_class": range(len(CLASS_IDS)),
        "category_id": CLASS_IDS,
        "name": [names.get(c, "") for c in CLASS_IDS],
    }
)

DATA_YAML = {mode: build_yolo_dataset(mode) / "data.yaml" for mode in DATA_CFG["split_modes"]}

# 만든 결과 점검: 사진 수 = 라벨 수, 클래스 번호가 범위 안인지
rows = []
for mode, yaml_path in DATA_YAML.items():
    out = yaml_path.parent
    for name in ["train", "val"]:
        imgs = list((out / "images" / name).iterdir())
        labels = list((out / "labels" / name).glob("*.txt"))
        lines = [ln for p in labels for ln in p.read_text(encoding="utf-8").splitlines()]
        cls = {int(ln.split()[0]) for ln in lines}
        assert len(imgs) == len(labels), f"{mode}/{name}: 사진과 라벨 수가 다르다"
        assert cls <= set(range(len(CLASS_IDS))), f"{mode}/{name}: 클래스 번호가 범위 밖"
        rows.append(
            {
                "분할 방식": mode,
                "구분": name,
                "사진": len(imgs),
                "박스": len(lines),
                "약 종류": len(cls),
            }
        )
print(f"클래스 {len(CLASS_IDS)}종 · 저장 위치: {PROCESSED_DIR / 'yolo'}")
pd.DataFrame(rows).set_index(["분할 방식", "구분"])


클래스 56종 · 저장 위치: /content/pill-bbox/data/processed/yolo


사진   박스  약 종류
분할 방식 구분                   
setup train  192  651    56
      val     40  120    17
combo train  194  651    56
      val     38  120    25

## 6. 학습: 실험 목록대로 돌리고 최고 모델 저장

**실험할 때 코드는 고치지 않고 `EXPERIMENTS` 목록에 한 줄만 추가한다.** 한 줄에는 이름, 분할 방식, 그리고 1단계 `TRAIN_CFG`에서 바꿀 값만 적는다. 적지 않은 값은 `TRAIN_CFG` 기본값을 쓴다.

| 스위치 | 뜻 |
|---|---|
| `RUN_TRAIN` | `True`로 바꿔야 학습이 시작된다 (실수로 긴 학습이 돌지 않게) |
| `QUICK_TEST` | `True`면 epoch 1번, 사진 25%만 써서 "에러 없이 돌아가는지"만 본다. 기록·저장 안 함 |
| `USE_WANDB` | wandb에 학습 곡선 기록 |
| `COPY_TO` | Colab이 끊겨도 결과가 남도록 Google Drive에 사본 저장 |

**저장되는 것** (`runs/`는 git에 올라가지 않음)

    runs/
    ├── train/<실험 이름>/          ultralytics 원본 결과 (곡선, 그림, weights)
    ├── best/<실험 이름>.pt         실험마다 가장 좋았던 가중치
    ├── best/overall_best_setup.pt  세팅 기준 실험 중 최고
    ├── best/overall_best_combo.pt  조합 기준 실험 중 최고
    └── experiments.csv             실험마다 한 줄: 설정 + 점수 (mAP50, mAP50-95, P, R, best epoch)

- 최고 모델은 **mAP50-95**로 고른다. 위치 정확도까지 따지는 더 엄격한 점수이고, 논문도 이 점수를 함께 보고했다.
- **세팅 기준과 조합 기준은 val이 서로 달라서 점수를 바로 비교하면 안 된다.** 그래서 최고 모델도 분할 방식마다 따로 둔다. 공정한 비교는 7단계에서 한다.
- `experiments.csv`에 이미 있는 이름은 건너뛴다. Colab이 끊겨 다시 실행해도 끝난 실험을 반복하지 않는다.


In [13]:
import gc
import time

import torch
from ultralytics import YOLO

from src.config import get_device, init_wandb

# 실험 목록: 이름, 분할 방식, 그리고 TRAIN_CFG에서 "바꿀 값만" 적는다
EXPERIMENTS = [
    {"name": "base_setup", "split": "setup"},
    {"name": "base_combo", "split": "combo"},
    # 예시: 분류 손실 비중 올리기 / 옵티마이저·학습률 직접 정하기 / 모델 크기 비교
    # {"name": "cls1.0_setup", "split": "setup", "cls": 1.0},
    # {"name": "adamw_1e-3_setup", "split": "setup", "optimizer": "AdamW", "lr0": 0.001},
    # {"name": "11s_setup", "split": "setup", "weights": "yolo11s.pt"},
]
RUN_TRAIN = False  # True로 바꿔야 학습이 시작된다 (실수로 긴 학습이 돌지 않게)
QUICK_TEST = True  # True면 epoch 1번, 사진 25%만 써서 "돌아가는지"만 본다. 기록·저장 안 함
USE_WANDB = True  # wandb에 학습 곡선 기록 (QUICK_TEST일 때는 끔)
COPY_TO = None  # Colab 끊김 대비 사본 위치. 예: Path("/content/drive/MyDrive/pill-bbox-runs")

BEST_DIR = RUNS_DIR / "best"  # 실험별 best 가중치 + 분할 방식별 최고 모델
RESULTS_CSV = RUNS_DIR / "experiments.csv"  # 실험마다 한 줄: 설정과 점수
SCORE = "mAP50-95"  # 최고 모델을 고르는 기준


def read_scores(save_dir: Path) -> dict:
    """ultralytics가 남긴 results.csv에서 mAP50-95가 가장 높았던 epoch의 점수를 읽는다."""
    log = pd.read_csv(save_dir / "results.csv")
    log.columns = log.columns.str.strip()
    best = log.loc[log["metrics/mAP50-95(B)"].idxmax()]
    return {
        "mAP50": round(float(best["metrics/mAP50(B)"]), 4),
        "mAP50-95": round(float(best["metrics/mAP50-95(B)"]), 4),
        "precision": round(float(best["metrics/precision(B)"]), 4),
        "recall": round(float(best["metrics/recall(B)"]), 4),
        "best_epoch": int(best["epoch"]),
        "epochs_run": len(log),
    }


def update_best(row: dict, best_pt: Path) -> bool:
    """이 실험이 같은 분할 방식에서 지금까지 최고 점수면 overall_best_<split>.pt로 저장한다."""
    done = pd.read_csv(RESULTS_CSV)
    same_split = done[done["split"] == row["split"]]
    if row[SCORE] < same_split[SCORE].max():
        return False
    shutil.copy2(best_pt, BEST_DIR / f"overall_best_{row['split']}.pt")
    return True


def train_one(name: str, split: str, **overrides) -> dict | None:
    """실험 하나: TRAIN_CFG에 overrides를 덮어 학습하고, 점수 기록과 best 가중치 저장까지 한다."""
    cfg = {**TRAIN_CFG, **overrides}
    weights = cfg.pop("weights")
    if QUICK_TEST:
        cfg.update(epochs=1, fraction=0.25, plots=False)
    model = YOLO(weights)
    run = None
    if USE_WANDB and not QUICK_TEST:
        run = init_wandb(model, run_name=name, config={"split": split, "weights": weights, **cfg})

    start = time.time()
    try:
        model.train(
            data=str(DATA_YAML[split]),
            project=str(RUNS_DIR / ("quick" if QUICK_TEST else "train")),
            name=name,
            device=get_device(),
            **cfg,
        )
    finally:
        if run is not None:
            run.finish()
    save_dir = Path(model.trainer.save_dir)
    if QUICK_TEST:
        print(f"[빠른 점검] {name}: 학습 정상 종료 → {save_dir}")
        return None

    best_pt = BEST_DIR / f"{name}.pt"
    shutil.copy2(save_dir / "weights" / "best.pt", best_pt)
    row = {
        "name": name,
        "split": split,
        **read_scores(save_dir),
        "weights": weights,
        "overrides": json.dumps(overrides, ensure_ascii=False),
        "seed": cfg["seed"],
        "fold": DATA_CFG["fold"],
        "minutes": round((time.time() - start) / 60, 1),
        "date": time.strftime("%Y-%m-%d %H:%M"),
        "save_dir": str(save_dir),
    }
    pd.DataFrame([row]).to_csv(RESULTS_CSV, mode="a", header=not RESULTS_CSV.exists(), index=False)
    row["new_best"] = update_best(row, best_pt)
    return row


if RUN_TRAIN:
    BEST_DIR.mkdir(parents=True, exist_ok=True)
    done_names = set(pd.read_csv(RESULTS_CSV)["name"]) if RESULTS_CSV.exists() else set()
    for exp in EXPERIMENTS:
        if exp["name"] in done_names and not QUICK_TEST:
            print(f"[건너뜀] {exp['name']}: 이미 기록이 있다 (다시 하려면 이름을 바꾼다)")
            continue
        row = train_one(**exp)
        if row:
            mark = " ← 새 최고!" if row["new_best"] else ""
            print(f"[완료] {row['name']} ({row['split']}): {SCORE} {row[SCORE]}{mark}")
        if COPY_TO is not None and not QUICK_TEST:
            shutil.copytree(BEST_DIR, Path(COPY_TO) / "best", dirs_exist_ok=True)
            shutil.copy2(RESULTS_CSV, Path(COPY_TO) / RESULTS_CSV.name)
        gc.collect()  # 다음 실험 전에 GPU 메모리를 비운다
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
else:
    print("RUN_TRAIN = False → 학습하지 않음. 실험 목록:")
    for exp in EXPERIMENTS:
        print(" -", exp)

if RESULTS_CSV.exists():
    print("\n지금까지의 실험 (점수 높은 순):")
    print(pd.read_csv(RESULTS_CSV).sort_values(SCORE, ascending=False).to_string(index=False))


RUN_TRAIN = False → 학습하지 않음. 실험 목록:
 - {'name': 'base_setup', 'split': 'setup'}
 - {'name': 'base_combo', 'split': 'combo'}
